# Chapter 01：Vector Add

**目标**：实现 `z = x + y`。本章只新增两个核心概念：一个 program 处理一块数据，以及用 mask 保护尾部越界访问。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name.startswith("chapter_"):
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import triton
import triton.language as tl

from common.benchmark import bench
from common.check import assert_close
from common.utils import get_device, set_seed

device = get_device()
set_seed(0)

## 1. PyTorch reference

PyTorch 表达式是正确性基准。使用非整除长度，确保稍后的 mask 真正被测试。

In [ ]:
n_elements = 1_000_003
x = torch.randn(n_elements, device=device)
y = torch.randn(n_elements, device=device)
torch_output = x + y
torch_output[:5]

## 2. Triton kernel

`tl.program_id(0)` 取得当前 program 编号。`tl.arange` 一次生成一组 offset，不是 Python `for` 循环。最后一个 program 可能超出 tensor 长度，所以 load/store 都使用 mask。

In [ ]:
@triton.jit
def vector_add_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    # Each program owns one block of consecutive elements.
    program_id = tl.program_id(axis=0)
    offsets = program_id * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x_values = tl.load(x_ptr + offsets, mask=mask)
    y_values = tl.load(y_ptr + offsets, mask=mask)
    tl.store(output_ptr + offsets, x_values + y_values, mask=mask)

## 3. Python wrapper 与 grid

`BLOCK_SIZE` 是每个 program 尝试处理的元素数。`grid = lambda meta: (...)` 在 launch 时读取 meta 参数，并计算需要多少个 program。

In [ ]:
def vector_add(x, y):
    if x.ndim != 1 or y.ndim != 1 or x.shape != y.shape:
        raise ValueError("x and y must be 1D tensors with the same shape")
    if not x.is_cuda or not y.is_cuda or not x.is_contiguous() or not y.is_contiguous():
        raise ValueError("x and y must be contiguous CUDA tensors")
    output = torch.empty_like(x)
    if x.numel() == 0:
        return output
    grid = lambda meta: (triton.cdiv(x.numel(), meta["BLOCK_SIZE"]),)
    vector_add_kernel[grid](x, y, output, x.numel(), BLOCK_SIZE=256)
    return output

## 4. Correctness check

长度 `1_000_003` 不能被 256 整除，因此这里也验证了尾部 mask。

In [ ]:
triton_output = vector_add(x, y)
assert_close("vector add", triton_output, torch_output)

## 5. Benchmark

`bench` 内部使用 `triton.testing.do_bench`，会正确处理 CUDA 异步执行。不要直接用 `time.time()` 包围 CUDA 操作。

In [ ]:
torch_ms = bench(lambda: x + y)
triton_ms = bench(lambda: vector_add(x, y))
print(f"PyTorch: {torch_ms:.3f} ms")
print(f"Triton:  {triton_ms:.3f} ms")

## 小结与练习

- grid 决定启动多少个 program。
- `BLOCK_SIZE` 决定每个 program 的 offset 向量长度。
- mask 让任意长度都能安全运行。

**练习**：把 `BLOCK_SIZE` 改为 128 或 512，重新检查正确性和 benchmark。